# 06.05_All_sc_og_processing_Python

九物种基因到 OG 表达映射。

- 当前文件：`analysis/06_single_cell_analysis/06.05_All_sc_og_processing_Python.ipynb`
- 原始来源：`Codes/06.05_sc_og_processing.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`anndata`, `matplotlib.pyplot`, `numpy`, `pandas`, `scanpy`, `seaborn`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


In [ ]:
fig_dir = '/share/home/zhangze/zz/NeuralOrigin/Figures'

## 数据基因替换为OG编号

### 1.1 以海葵数据为例测试

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata

In [ ]:
# 海葵 Neve
adata_Neve = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/Neve.normalized.h5ad")
adata_Neve

In [ ]:
adata_Neve.var

In [ ]:
# 保留原始 gene_id
adata_Neve.var['raw_gene_id'] = adata_Neve.var_names
adata_Neve.var

In [ ]:
# 读入 og 文件，映射 orthogroup
og_Neve = pd.read_csv('/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/Neve.protein_to_orthogroup.csv')  # 两列：protein_id, orthogroup
og_Neve_map = og_Neve.set_index('protein_id')['orthogroup'].to_dict()
og_Neve_map

In [ ]:
# 映射 OG
# 创建新列：若有OG则写入，否则保留原名
adata_Neve.var['orthogroup'] = adata_Neve.var_names.map(lambda x: og_Neve_map[x] if x in og_Neve_map else x)
adata_Neve.var

In [ ]:
type(adata_Neve.X)

In [ ]:
# 将表达矩阵转为 dataframe
expr_df = pd.DataFrame(adata_Neve.X, columns=adata_Neve.var_names, index=adata_Neve.obs_names)
expr_df.columns = adata_Neve.var['orthogroup'].values  # 替换为OG

# 聚合表达（按列，也就是合并相同OG）
expr_og_df = expr_df.groupby(expr_df.columns, axis=1).sum()

In [ ]:
adata_og = anndata.AnnData(
    X=expr_og_df.values, 
    obs=adata_Neve.obs.copy(), 
    var=pd.DataFrame(index=expr_og_df.columns))
adata_og.uns = adata_Neve.uns.copy()
adata_og.obsm = adata_Neve.obsm.copy()
adata_og

In [ ]:
adata_og.var

### 1.2 映射方法

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata

def adata_to_orthogroup(
    adata_path,
    og_map_path,
    save_path=None
):
    """
    将AnnData对象的基因映射为orthogroup，并合并同OG表达（均值），返回新AnnData对象。
    :param adata_path: 原始h5ad文件路径
    :param og_map_path: 两列(protein_id, orthogroup)的OG映射csv路径
    :param save_path: 若不为None，则保存为此h5ad
    :return: 新的OG级别AnnData对象
    """
    print(f"\n===== 处理 {adata_path} =====")
    # 1. 读取AnnData
    adata = sc.read_h5ad(adata_path)
    print(f"原始基因数量：{adata.n_vars}")
    # 2. 保存原始gene_id
    adata.var['raw_gene_id'] = adata.var_names
    # 3. 加载OG映射
    og_df = pd.read_csv(og_map_path)
    og_map = og_df.set_index('protein_id')['orthogroup'].to_dict()
    print(f"OG映射表中OG数量：{len(set(og_map.values()))}，蛋白总数：{len(og_map)}")
    # 4. 建orthogroup列
    adata.var['orthogroup'] = adata.var_names.map(lambda x: og_map[x] if x in og_map else x)
    n_mapped = sum(~adata.var['orthogroup'].eq(adata.var['raw_gene_id']))
    n_unmapped = sum(adata.var['orthogroup'].eq(adata.var['raw_gene_id']))
    print(f"能被映射到OG的基因数：{n_mapped}，未能映射的：{n_unmapped}")
    # 5. 表达聚合
    expr_df = pd.DataFrame(
        adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X,
        columns=adata.var_names,
        index=adata.obs_names
    )
    expr_df.columns = adata.var['orthogroup'].values
    og_before = expr_df.shape[1]
    # 聚合时可以考虑sum或mean
    expr_og_df = expr_df.groupby(expr_df.columns, axis=1).sum()
    og_after = expr_og_df.shape[1]
    print(f"整合前OG维度（含重复）：{og_before}，整合后唯一OG数量：{og_after}")
    # 6. 构建新AnnData
    adata_og = anndata.AnnData(
        X=expr_og_df.values,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=expr_og_df.columns)
    )
    adata_og.uns = adata.uns.copy()
    adata_og.obsm = adata.obsm.copy()
    if save_path is not None:
        adata_og.write_h5ad(save_path)
        print(f"已保存至：{save_path}")
    print(f"新 AnnData: cells = {adata_og.n_obs}，OG = {adata_og.n_vars}\n")
    return adata_og

### 1.3 开始映射

In [ ]:
# 斑马鱼 Dare
sp = 'Dare'
adata_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/{sp}.normalized.h5ad"
og_map_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/{sp}.protein_to_orthogroup.csv"

adata_Dare = sc.read_h5ad(adata_path)
print(adata_Dare)
adata_Dare_og = adata_to_orthogroup(adata_path, og_map_path)
print(adata_Dare_og)

In [ ]:
# 海葵 Neve
sp = 'Neve'
adata_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/{sp}.normalized.h5ad"
og_map_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/{sp}.protein_to_orthogroup.csv"
# save_path = f"/path/to/{sp}_og.h5ad"

adata_Neve = sc.read_h5ad(adata_path)
print(adata_Neve)
adata_Neve_og = adata_to_orthogroup(adata_path, og_map_path)
print(adata_Neve_og)

In [ ]:
# 半球美螅水母 Clhe
sp = 'Clhe'
adata_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/{sp}.normalized.h5ad"
og_map_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/{sp}.protein_to_orthogroup.csv"
# save_path = f"/path/to/{sp}_og.h5ad"

adata_Clhe = sc.read_h5ad(adata_path)
print(adata_Clhe)
adata_Clhe_og = adata_to_orthogroup(adata_path, og_map_path)
print(adata_Clhe_og)

In [ ]:
# 海月水母 Auco
sp = 'Auco'
# adata_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/{sp}.normalized.h5ad"
adata_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellAnnotation/Auco.normalized.annotated.h5ad"
og_map_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/{sp}.protein_to_orthogroup.csv"
# save_path = f"/path/to/{sp}_og.h5ad"

adata_Auco = sc.read_h5ad(adata_path)
print(adata_Auco)
adata_Auco_og = adata_to_orthogroup(adata_path, og_map_path)
print(adata_Auco_og)

In [ ]:
# 丝盘虫 TrH1
sp = 'TrH1'
adata_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/{sp}.normalized.h5ad"
og_map_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/{sp}.protein_to_orthogroup.csv"
# save_path = f"/path/to/{sp}_og.h5ad"

adata_TrH1 = sc.read_h5ad(adata_path)
print(adata_TrH1)
adata_TrH1_og = adata_to_orthogroup(adata_path, og_map_path)
print(adata_TrH1_og)

In [ ]:
# 丝盘虫 TrH2
sp = 'TrH2'
adata_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/{sp}.normalized.h5ad"
og_map_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/{sp}.protein_to_orthogroup.csv"
# save_path = f"/path/to/{sp}_og.h5ad"

adata_TrH2 = sc.read_h5ad(adata_path)
print(adata_TrH2)
adata_TrH2_og = adata_to_orthogroup(adata_path, og_map_path)
print(adata_TrH2_og)

In [ ]:
# 丝盘虫 HoH13
sp = 'HoH13'
adata_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/{sp}.normalized.h5ad"
og_map_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/{sp}.protein_to_orthogroup.csv"
# save_path = f"/path/to/{sp}_og.h5ad"

adata_HoH13 = sc.read_h5ad(adata_path)
print(adata_HoH13)
adata_HoH13_og = adata_to_orthogroup(adata_path, og_map_path)
print(adata_HoH13_og)

In [ ]:
# 丝盘虫 ClH23
sp = 'ClH23'
adata_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/{sp}.normalized.h5ad"
og_map_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/{sp}.protein_to_orthogroup.csv"
# save_path = f"/path/to/{sp}_og.h5ad"

adata_ClH23 = sc.read_h5ad(adata_path)
print(adata_ClH23)
adata_ClH23_og = adata_to_orthogroup(adata_path, og_map_path)
print(adata_ClH23_og)

In [ ]:
# 海绵 Spla
sp = 'Spla'
adata_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/{sp}.normalized.h5ad"
og_map_path = f"/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/{sp}.protein_to_orthogroup.csv"
# save_path = f"/path/to/{sp}_og.h5ad"

adata_Spla = sc.read_h5ad(adata_path)
print(adata_Spla)
adata_Spla_og = adata_to_orthogroup(adata_path, og_map_path)
print(adata_Spla_og)

## 2.导出数据

In [ ]:
# 特有基因
print(len(set(adata_Dare_og.var_names)))
print(len(set(adata_Neve_og.var_names)))
print(len(set(adata_Clhe_og.var_names)))
print(len(set(adata_Auco_og.var_names)))
print(len(set(adata_TrH1_og.var_names)))
print(len(set(adata_TrH2_og.var_names)))
print(len(set(adata_HoH13_og.var_names)))
print(len(set(adata_ClH23_og.var_names)))
print(len(set(adata_Spla_og.var_names)))

In [ ]:
# 共有基因统计
common_ogs = set(adata_Dare_og.var_names) & set(adata_Neve_og.var_names)
print(len(common_ogs))
common_ogs = common_ogs & set(adata_Clhe_og.var_names)
print(len(common_ogs))
common_ogs = common_ogs & set(adata_Auco_og.var_names)
print(len(common_ogs))
common_ogs = common_ogs & set(adata_TrH1_og.var_names)
print(len(common_ogs))
common_ogs = common_ogs & set(adata_TrH2_og.var_names)
print(len(common_ogs))
common_ogs = common_ogs & set(adata_HoH13_og.var_names)
print(len(common_ogs))
common_ogs = common_ogs & set(adata_ClH23_og.var_names)
print(len(common_ogs))
common_ogs = common_ogs & set(adata_Spla_og.var_names)
print(len(common_ogs))

In [ ]:
import matplotlib.pyplot as plt

# 物种顺序（请按实际顺序修改）
species = [
    'Dare', 'Neve', 'Clhe', 'Auco', 'TrH1', 'TrH2', 'HoH13', 'ClH23', 'Spla'
]

# 每个物种的OG总数（按顺序）
total_ogs = [27883, 13249, 38770, 10867, 11509, 9663, 9923, 10809, 21251]

# 每步的共有OG数（你上面统计的，长度比total_ogs少1，可补第一个物种时的初值）
common_ogs = [4854, 3487, 3298, 2974, 2951, 2817, 2805, 2216]
# 补上第一个物种本身的OG总数（这样两条线长度一致）
common_ogs_full = [total_ogs[0]] + common_ogs  # [14721, 5383, 4820, ...]

x = range(len(species))

fig, ax1 = plt.subplots(figsize=(10,6))

# 第一条y轴：各物种OG总数
ax1.plot(x, total_ogs, marker='o', color='tab:blue', label='Total OGs')
ax1.set_ylabel('Total OGs in Each Species', color='tab:blue', fontsize=12)
ax1.tick_params(axis='y', labelcolor='tab:blue')

# 第二条y轴：逐步共有OG数
ax2 = ax1.twinx()
ax2.plot(x, common_ogs_full, marker='s', color='tab:red', label='Common OGs (cumulative)')
ax2.set_ylabel('Cumulative Common OGs', color='tab:red', fontsize=12)
ax2.tick_params(axis='y', labelcolor='tab:red')

# 横坐标和美化
plt.xticks(x, species, rotation=45, fontsize=12)
plt.title('OG Statistics: Total vs. Common (Cumulative) by Species', fontsize=14)
fig.tight_layout()
fig.legend(loc='upper right', bbox_to_anchor=(0.87, 0.87))
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()


### 2.1 斑马鱼

In [ ]:
# 斑马鱼
print(adata_Dare)
print(adata_Dare_og)

In [ ]:
# 只保留CellType
adata_Dare_og.obs = adata_Dare_og.obs[['broad_cell_type']].copy()
# 只保留 X_umap（如果有多维的obsm，需手动清理其它项）
adata_Dare_og.obsm = {'X_umap': adata_Dare_og.obsm['X_umap'].copy()}
adata_Dare_og

In [ ]:
adata_Dare_og.obs['CellTypes'] = 'Dare_' + adata_Dare_og.obs['broad_cell_type'].astype(str)

In [ ]:
# 清空
adata_Dare_og.uns = {}
adata_Dare_og

In [ ]:
# 修改细胞名
adata_Dare_og.obs.index.name = "CellName"
adata_Dare_og.obs

In [ ]:
# 修改基因索引名
adata_Dare_og.var.index.name = "OrthoGene"
adata_Dare_og.var

In [ ]:
sc.pl.umap(adata_Dare_og, color=['CellTypes'])

In [ ]:
import scanpy as sc

# 1. 定义您的颜色字典
dare_color_dict = {
    "Dare_Endoderm": "#F3C4FF",
    "Dare_Epidermal": "#BC8EE7",
    "Dare_Mesoderm": "#9467BD",
    "Dare_Neural Anterior": "#8EFC89",
    "Dare_Neural Crest": "#58C755",
    "Dare_Neural Mid": "#2ca02c",
    "Dare_Neural Posterior": "#007800",
    "Dare_Germline": "#fedb61"
}

# 2. 确保 CellType 是 categorical 类型并获取其顺序
adata_Dare_og.obs['CellTypes'] = adata_Dare_og.obs['CellTypes'].astype('category')
categories = adata_Dare_og.obs['CellTypes'].cat.categories

# 3. 按照类别顺序生成颜色列表
new_colors = [dare_color_dict.get(cat, "#cccccc") for cat in categories]

# 4. 写入 adata 对象的 uns 属性
adata_Dare_og.uns['CellTypes_colors'] = new_colors

# 5. 绘图
sc.pl.umap(adata_Dare_og, color=['CellTypes'], title="Zebrafish (Dare) Cell Types", legend_loc='on data')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


fig, ax = plt.subplots(dpi=300)

sc.pl.umap(
    adata_Dare_og,
    color='CellTypes',
    ax=ax,
    show=False,
    legend_loc=None,   # ⭐ 不显示图例
    frameon=False      # ⭐ 去掉坐标轴边框
)

# ⭐ 去掉坐标刻度与标签、标题
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

output_path = fig_dir + "/26.UMAP.Dare.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
adata_Dare_og

In [ ]:
adata_Dare_og.obs['CellTypes'].value_counts()

In [ ]:
# 保存adata数据
raw_Dare_path = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Dare.OG.normalized.h5ad"
adata_Dare_og.write(raw_Dare_path)

In [ ]:
# 导出基因id
adata_Dare_og.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Dare.OG.genes.txt', index=False, header=False)

### 2.2 海葵

In [ ]:
# 海葵
print(adata_Neve)
print(adata_Neve_og)

In [ ]:
# 只保留CellType
adata_Neve_og.obs = adata_Neve_og.obs[['CellType']].copy()
# 只保留 X_umap（如果有多维的obsm，需手动清理其它项）
adata_Neve_og.obsm = {'X_umap': adata_Neve_og.obsm['X_umap'].copy()}
adata_Neve_og

In [ ]:
adata_Neve_og.obs['CellTypes'] = 'Neve_' + adata_Neve_og.obs['CellType'].astype(str)

In [ ]:
# 修改细胞名
adata_Neve_og.obs.index.name = "CellName"
adata_Neve_og.obs

In [ ]:
# 修改基因索引名
adata_Neve_og.var.index.name = "OrthoGene"
adata_Neve_og.var

In [ ]:
sc.pl.umap(adata_Neve_og, color=['CellTypes'])

In [ ]:
import scanpy as sc

# 1. 定义 Neve 的颜色字典
neve_color_dict = {
    # 外胚层与内胚皮层 (紫色系梯度)
    "Neve_ectoderm.embryonic": "#F3C4FF",
    "Neve_ectoderm.epidermis": "#BC8EE7",
    "Neve_ectoderm.pharyngeal": "#9467BD",
    "Neve_retractor muscle": "#4C1E6F",
    
    # 胃皮层 (橙色)
    "Neve_gastrodermis": "#ff7f0e",
    
    # 神经系统 (绿色系)
    "Neve_NPC": "#8EFC89",
    "Neve_neuronal": "#2ca02c",
    
    # 腺体与分泌细胞 (红色系)
    "Neve_gland.mucous": "#d62728",
    "Neve_secretory": "#FF9B8C",
    
    # 刺细胞 (青色系)
    "Neve_cnidocyte": "#46D9EA",
    "Neve_cnidocyte.mature": "#17becf",

    # 干细胞
    "Neve_mesendoderm.embryonic": "#fedb61"
}

# 2. 确保 CellTypes 是 categorical 类型
adata_Neve_og.obs['CellTypes'] = adata_Neve_og.obs['CellTypes'].astype('category')
categories = adata_Neve_og.obs['CellTypes'].cat.categories

# 3. 按照类别顺序生成颜色列表，未定义的默认为灰色
new_colors_neve = [neve_color_dict.get(cat, "#cccccc") for cat in categories]

# 4. 写入 adata 对象的 uns 属性 (注意属性名需与 obs 中的列名对应)
adata_Neve_og.uns['CellTypes_colors'] = new_colors_neve

# 5. 绘图
sc.pl.umap(adata_Neve_og, color=['CellTypes'], title="Nematostella (Neve) Cell Types", legend_loc='on data')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


fig, ax = plt.subplots(dpi=300)

sc.pl.umap(
    adata_Neve_og,
    color='CellTypes',
    ax=ax,
    show=False,
    legend_loc=None,   # ⭐ 不显示图例
    frameon=False      # ⭐ 去掉坐标轴边框
)

# ⭐ 去掉坐标刻度与标签、标题
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

output_path = fig_dir + "/27.UMAP.Neve.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
adata_Neve_og

In [ ]:
adata_Neve_og.obs['CellTypes'].value_counts()

In [ ]:
# 保存adata数据
raw_Neve_path = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Neve.OG.normalized.h5ad"
adata_Neve_og.write(raw_Neve_path)

In [ ]:
# 导出基因id
adata_Neve_og.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Neve.OG.genes.txt', index=False, header=False)

### 2.3 半球美螅水母

In [ ]:
# 半球美螅水母
print(adata_Clhe)
print(adata_Clhe_og)

In [ ]:
# 只保留CellType
adata_Clhe_og.obs = adata_Clhe_og.obs[['annos']].copy()
# 只保留 X_umap（如果有多维的obsm，需手动清理其它项）
adata_Clhe_og.obsm = {'X_umap': adata_Clhe_og.obsm['X_umap'].copy()}
adata_Clhe_og

In [ ]:
adata_Clhe_og.obs['CellTypes'] = 'Clhe_' + adata_Clhe_og.obs['annos'].astype(str)

In [ ]:
# 清空
adata_Clhe_og.uns = {}
adata_Clhe_og

In [ ]:
# 修改细胞名
adata_Clhe_og.obs.index.name = "CellName"
adata_Clhe_og.obs

In [ ]:
# 修改基因索引名
adata_Clhe_og.var.index.name = "OrthoGene"
adata_Clhe_og.var

In [ ]:
sc.pl.umap(adata_Clhe_og, color=['CellTypes'])

In [ ]:
import scanpy as sc

# 1. 定义 Clhe 的颜色字典 (严格对应您提供的色值)
clhe_color_dict = {
    "Clhe_Epidermal/Muscle": "#9467bd",     # 紫色
    "Clhe_Gastroderm": "#ff7f0e",           # 橙色
    "Clhe_Neural": "#2ca02c",               # 棕色
    "Clhe_Gland Cell": "#d62728",           # 红色
    "Clhe_Stem Cell/Germ Cell": "#fedb61",  # 黄色
    "Clhe_Nematocyte": "#17becf",           # 青色
    "Clhe_Bioluminescent Cells": "#000000"  # 黑色
}

# 2. 确保 CellType 是 categorical 类型并获取其顺序
# 注意：基于您的描述使用 adata_Clhe_og 对象
adata_Clhe_og.obs['CellTypes'] = adata_Clhe_og.obs['CellTypes'].astype('category')
categories = adata_Clhe_og.obs['CellTypes'].cat.categories

# 3. 按照类别顺序生成颜色列表
new_colors_clhe = [clhe_color_dict.get(cat, "#cccccc") for cat in categories]

# 4. 写入 adata 对象的 uns 属性
# 规则：uns 中的键名必须是 {obs_column}_colors
adata_Clhe_og.uns['CellTypes_colors'] = new_colors_clhe

# 5. 绘图
sc.pl.umap(
    adata_Clhe_og, 
    color=['CellTypes'], 
    title="Clytia (Clhe) Cell Types", 
    legend_loc='on data',
    frameon=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


fig, ax = plt.subplots(dpi=300)

sc.pl.umap(
    adata_Clhe_og,
    color='CellTypes',
    ax=ax,
    show=False,
    legend_loc=None,   # ⭐ 不显示图例
    frameon=False      # ⭐ 去掉坐标轴边框
)

# ⭐ 去掉坐标刻度与标签、标题
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

output_path = fig_dir + "/28.UMAP.Clhe.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
adata_Clhe_og

In [ ]:
adata_Clhe_og.obs['CellTypes'].value_counts()

In [ ]:
# 保存adata数据
raw_Clhe_path = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Clhe.OG.normalized.h5ad"
adata_Clhe_og.write(raw_Clhe_path)

In [ ]:
# 导出基因id
adata_Clhe_og.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Clhe.OG.genes.txt', index=False, header=False)

### 2.4 海月水母

In [ ]:
# 海月水母
print(adata_Auco)
print(adata_Auco_og)

In [ ]:
# 保留CellType、CellType_v1、CellType_v2
adata_Auco_og.obs = adata_Auco_og.obs[['seurat_clusters', 'Sub_cell_type', 'Broad_cell_type']].copy()
# adata_Auco_og.obs['CellType'] = adata_Auco_og.obs['seurat_clusters'].copy()
# 只保留 X_umap（如果有多维的obsm，需手动清理其它项）
adata_Auco_og.obsm = {'X_umap': adata_Auco_og.obsm['X_umap'].copy()}
adata_Auco_og

In [ ]:
adata_Auco_og.obs['CellTypes'] = 'Auco_' + adata_Auco_og.obs['Broad_cell_type'].astype(str)

In [ ]:
# 清空
adata_Auco_og.uns = {}
adata_Auco_og

In [ ]:
# 修改细胞名
adata_Auco_og.obs.index.name = "CellName"
adata_Auco_og.obs

In [ ]:
# 修改基因索引名
adata_Auco_og.var.index.name = "OrthoGene"
adata_Auco_og.var

In [ ]:
sc.pl.umap(adata_Auco_og, color=['CellTypes'])

In [ ]:
import scanpy as sc

# 1. 定义您的颜色字典
auco_color_dict = {
    'Auco_CN, Cnidocytes/Nematocyte cell': '#17becf', # CN 青色 '#17becf'
    'Auco_EM, Epidermal/Muscle cell': '#9467bd',      # EM 紫色 '#9467bd'
    'Auco_GA, Gastrodermal cell': '#ff7f0e',          # GA 橙色 '#ff7f0e'
    'Auco_GL, Gland cell': '#d62728',                 # GL 红色 '#d62728'
    'Auco_HA, Hair cell': '#8c564b',                  # HA 棕色 '#8c564b'
    'Auco_NE, Neural cell': '#2ca02c',                # NE 绿色 '#2ca02c'
    'Auco_SG, Stem/Germline cell': '#fedb61'          # SG 黄色 '#fedb61'
}

# 2. 确保 CellType 是 categorical 类型并获取其顺序
adata_Auco_og.obs['CellTypes'] = adata_Auco_og.obs['CellTypes'].astype('category')
categories = adata_Auco_og.obs['CellTypes'].cat.categories

# 3. 按照类别顺序生成颜色列表
new_colors = [auco_color_dict.get(cat, "#cccccc") for cat in categories]

# 4. 写入 adata 对象的 uns 属性
adata_Auco_og.uns['CellTypes_colors'] = new_colors

# 5. 绘图
sc.pl.umap(adata_Auco_og, color=['CellTypes'], title="Auco", legend_loc='on data')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


fig, ax = plt.subplots(dpi=300)

sc.pl.umap(
    adata_Auco_og,
    color='CellTypes',
    ax=ax,
    show=False,
    legend_loc=None,   # ⭐ 不显示图例
    frameon=False      # ⭐ 去掉坐标轴边框
)

# ⭐ 去掉坐标刻度与标签、标题
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

output_path = fig_dir + "/29.UMAP.Auco.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
adata_Auco_og

In [ ]:
adata_Auco_og.obs['CellTypes'].value_counts()

In [ ]:
# 保存adata数据
raw_Auco_path = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Auco.OG.normalized.h5ad"
adata_Auco_og.write(raw_Auco_path)

In [ ]:
# 导出基因id
adata_Auco_og.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Auco.OG.genes.txt', index=False, header=False)

### 2.5 丝盘虫TrH1

In [ ]:
# 半球美螅水母
print(adata_TrH1)
print(adata_TrH1_og)

In [ ]:
# 只保留CellType
adata_TrH1_og.obs = adata_TrH1_og.obs[['broad_cell_type']].copy()
# 只保留 X_umap（如果有多维的obsm，需手动清理其它项）
adata_TrH1_og.obsm = {'X_umap': adata_TrH1_og.obsm['X_umap'].copy()}
adata_TrH1_og

In [ ]:
adata_TrH1_og.obs['CellTypes'] = 'TrH1_' + adata_TrH1_og.obs['broad_cell_type'].astype(str)

In [ ]:
# 清空
adata_TrH1_og.uns = {}
adata_TrH1_og

In [ ]:
# 修改细胞名
adata_TrH1_og.obs.index.name = "CellName"
adata_TrH1_og.obs

In [ ]:
# 修改基因索引名
adata_TrH1_og.var.index.name = "OrthoGene"
adata_TrH1_og.var

In [ ]:
sc.pl.umap(adata_TrH1_og, color=['CellTypes'])

In [ ]:
import scanpy as sc

# 1. 定义 TrH1 的颜色字典 (严格对应您提供的色值)
trh1_color_dict = {
    # 上皮与纤维细胞 (紫色系)
    "TrH1_epithelia": "#F3C4FF",
    "TrH1_epithelia_gland_like": "#BC8EE7",
    "TrH1_fibre": "#9467BD",
    
    # 肽能细胞 (绿色)
    "TrH1_peptidergic": "#2ca02c",
    
    # 腺体与脂肪细胞 (红色系)
    "TrH1_gland": "#FF9B8C",
    "TrH1_lipophil": "#d62728",
    
    # 减数分裂细胞 (黄色)
    "TrH1_meiotic": "#fedb61",
    
    # 过渡态细胞 (黑色)
    "TrH1_trans": "#000000"
}

# 2. 确保 CellType 是 categorical 类型并获取其顺序
# 使用 adata_TrH1_og 对象进行操作
adata_TrH1_og.obs['CellTypes'] = adata_TrH1_og.obs['CellTypes'].astype('category')
categories = adata_TrH1_og.obs['CellTypes'].cat.categories

# 3. 按照类别顺序生成颜色列表
new_colors_trh1 = [trh1_color_dict.get(cat, "#cccccc") for cat in categories]

# 4. 写入 adata 对象的 uns 属性
adata_TrH1_og.uns['CellTypes_colors'] = new_colors_trh1

# 5. 绘图
sc.pl.umap(
    adata_TrH1_og, 
    color=['CellTypes'], 
    title="Trichoplax (TrH1) Cell Types", 
    legend_loc='on data',
    frameon=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


fig, ax = plt.subplots(dpi=300)

sc.pl.umap(
    adata_TrH1_og,
    color='CellTypes',
    ax=ax,
    show=False,
    legend_loc=None,   # ⭐ 不显示图例
    frameon=False      # ⭐ 去掉坐标轴边框
)

# ⭐ 去掉坐标刻度与标签、标题
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

output_path = fig_dir + "/30.UMAP.TrH1.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
adata_TrH1_og

In [ ]:
adata_TrH1_og.obs['CellTypes'].value_counts()

In [ ]:
# 保存adata数据
raw_TrH1_path = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/TrH1.OG.normalized.h5ad"
adata_TrH1_og.write(raw_TrH1_path)

In [ ]:
# 导出基因id
adata_TrH1_og.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/TrH1.OG.genes.txt', index=False, header=False)

### 2.6 丝盘虫TrH2

In [ ]:
# 半球美螅水母
print(adata_TrH2)
print(adata_TrH2_og)

In [ ]:
# 只保留CellType
adata_TrH2_og.obs = adata_TrH2_og.obs[['broad_cell_type']].copy()
# 只保留 X_umap（如果有多维的obsm，需手动清理其它项）
adata_TrH2_og.obsm = {'X_umap': adata_TrH2_og.obsm['X_umap'].copy()}
adata_TrH2_og

In [ ]:
adata_TrH2_og.obs['CellTypes'] = 'TrH2_' + adata_TrH2_og.obs['broad_cell_type'].astype(str)

In [ ]:
# 清空
adata_TrH2_og.uns = {}
adata_TrH2_og

In [ ]:
# 修改细胞名
adata_TrH2_og.obs.index.name = "CellName"
adata_TrH2_og.obs

In [ ]:
# 修改基因索引名
adata_TrH2_og.var.index.name = "OrthoGene"
adata_TrH2_og.var

In [ ]:
sc.pl.umap(adata_TrH2_og, color=['CellTypes'])

In [ ]:
import scanpy as sc

# 1. 定义 TrH2 的颜色字典
trh2_color_dict = {
    # 上皮与纤维细胞 (紫色系)
    "TrH2_epithelia": "#F3C4FF",
    "TrH2_epithelia_gland_like": "#BC8EE7",
    "TrH2_fibre": "#9467BD",
    
    # 肽能细胞 (绿色)
    "TrH2_peptidergic": "#2ca02c",
    
    # 腺体与脂肪细胞 (红色系)
    "TrH2_gland": "#FF9B8C",
    "TrH2_lipophil": "#d62728",
    
    # 减数分裂细胞 (黄色)
    "TrH2_meiotic": "#fedb61",
    
    # 过渡态细胞 (黑色)
    "TrH2_trans": "#000000",
    
    # 未知类群 (深灰色)
    "TrH2_unknown_1": "#707070"
}

# 2. 确保 CellType 是 categorical 类型并获取其顺序
# 使用 adata_TrH2_og 对象
adata_TrH2_og.obs['CellTypes'] = adata_TrH2_og.obs['CellTypes'].astype('category')
categories = adata_TrH2_og.obs['CellTypes'].cat.categories

# 3. 按照类别顺序生成颜色列表
new_colors_trh2 = [trh2_color_dict.get(cat, "#cccccc") for cat in categories]

# 4. 写入 adata 对象的 uns 属性
adata_TrH2_og.uns['CellTypes_colors'] = new_colors_trh2

# 5. 绘图
sc.pl.umap(
    adata_TrH2_og, 
    color=['CellTypes'], 
    title="Trichoplax H2 (TrH2) Cell Types", 
    legend_loc='on data',
    frameon=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


fig, ax = plt.subplots(dpi=300)

sc.pl.umap(
    adata_TrH2_og,
    color='CellTypes',
    ax=ax,
    show=False,
    legend_loc=None,   # ⭐ 不显示图例
    frameon=False      # ⭐ 去掉坐标轴边框
)

# ⭐ 去掉坐标刻度与标签、标题
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

output_path = fig_dir + "/31.UMAP.TrH2.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
adata_TrH2_og

In [ ]:
adata_TrH2_og.obs['CellTypes'].value_counts()

In [ ]:
# 保存adata数据
raw_TrH2_path = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/TrH2.OG.normalized.h5ad"
adata_TrH2_og.write(raw_TrH2_path)

In [ ]:
# 导出基因id
adata_TrH2_og.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/TrH2.OG.genes.txt', index=False, header=False)

### 2.7 丝盘虫HoH13

In [ ]:
# 半球美螅水母
print(adata_HoH13)
print(adata_HoH13_og)

In [ ]:
# 只保留CellType
adata_HoH13_og.obs = adata_HoH13_og.obs[['broad_cell_type']].copy()
# 只保留 X_umap（如果有多维的obsm，需手动清理其它项）
adata_HoH13_og.obsm = {'X_umap': adata_HoH13_og.obsm['X_umap'].copy()}
adata_HoH13_og

In [ ]:
adata_HoH13_og.obs['CellTypes'] = 'HoH13_' + adata_HoH13_og.obs['broad_cell_type'].astype(str)

In [ ]:
# 清空
adata_HoH13_og.uns = {}
adata_HoH13_og

In [ ]:
# 修改细胞名
adata_HoH13_og.obs.index.name = "CellName"
adata_HoH13_og.obs

In [ ]:
# 修改基因索引名
adata_HoH13_og.var.index.name = "OrthoGene"
adata_HoH13_og.var

In [ ]:
sc.pl.umap(adata_HoH13_og, color=['CellTypes'])

In [ ]:
import scanpy as sc

# 1. 定义 HoH13 的颜色字典
hoh13_color_dict = {
    # 上皮与纤维细胞 (紫色系)
    "HoH13_epithelia": "#F3C4FF",
    "HoH13_epithelia_gland_like": "#BC8EE7",
    "HoH13_fibre": "#9467BD",
    
    # 肽能细胞 (绿色) - 神经起源锚点
    "HoH13_peptidergic": "#2ca02c",
    
    # 腺体与脂肪细胞 (红色系)
    "HoH13_gland": "#FF9B8C",
    "HoH13_lipophil": "#d62728",
    
    # 过渡态细胞 (黑色)
    "HoH13_trans": "#000000",
    
    # 未知类群 (深灰色)
    "HoH13_unknown_1": "#707070"
}

# 2. 确保 CellType 是 categorical 类型
# 使用 adata_HoH13_og 对象
adata_HoH13_og.obs['CellTypes'] = adata_HoH13_og.obs['CellTypes'].astype('category')
categories = adata_HoH13_og.obs['CellTypes'].cat.categories

# 3. 按照类别顺序生成颜色列表
new_colors_hoh13 = [hoh13_color_dict.get(cat, "#cccccc") for cat in categories]

# 4. 写入 adata 对象的 uns 属性
adata_HoH13_og.uns['CellTypes_colors'] = new_colors_hoh13

# 5. 绘图
sc.pl.umap(
    adata_HoH13_og, 
    color=['CellTypes'], 
    title="Hoilungia (HoH13) Cell Types", 
    legend_loc='on data',
    frameon=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


fig, ax = plt.subplots(dpi=300)

sc.pl.umap(
    adata_HoH13_og,
    color='CellTypes',
    ax=ax,
    show=False,
    legend_loc=None,   # ⭐ 不显示图例
    frameon=False      # ⭐ 去掉坐标轴边框
)

# ⭐ 去掉坐标刻度与标签、标题
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

output_path = fig_dir + "/32.UMAP.HoH13.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
adata_HoH13_og

In [ ]:
adata_HoH13_og.obs['CellTypes'].value_counts()

In [ ]:
# 保存adata数据
raw_HoH13_path = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/HoH13.OG.normalized.h5ad"
adata_HoH13_og.write(raw_HoH13_path)

In [ ]:
# 导出基因id
adata_HoH13_og.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/HoH13.OG.genes.txt', index=False, header=False)

### 2.8 丝盘虫ClH23

In [ ]:
# 半球美螅水母
print(adata_ClH23)
print(adata_ClH23_og)

In [ ]:
# 只保留CellType
adata_ClH23_og.obs = adata_ClH23_og.obs[['broad_cell_type']].copy()
# 只保留 X_umap（如果有多维的obsm，需手动清理其它项）
adata_ClH23_og.obsm = {'X_umap': adata_ClH23_og.obsm['X_umap'].copy()}
adata_ClH23_og

In [ ]:
adata_ClH23_og.obs['CellTypes'] = 'ClH23_' + adata_ClH23_og.obs['broad_cell_type'].astype(str)

In [ ]:
# 清空
adata_ClH23_og.uns = {}
adata_ClH23_og

In [ ]:
# 修改细胞名
adata_ClH23_og.obs.index.name = "CellName"
adata_ClH23_og.obs

In [ ]:
# 修改基因索引名
adata_ClH23_og.var.index.name = "OrthoGene"
adata_ClH23_og.var

In [ ]:
sc.pl.umap(adata_ClH23_og, color=['CellTypes'])

In [ ]:
import scanpy as sc

# 1. 定义 ClH23 的颜色字典
clh23_color_dict = {
    # 上皮与纤维细胞 (紫色系)
    "ClH23_epithelia": "#F3C4FF",
    "ClH23_epithelia_gland_like": "#BC8EE7",
    "ClH23_fibre": "#9467BD",
    
    # 肽能细胞 (绿色) - 跨物种神经演化锚点
    "ClH23_peptidergic": "#2ca02c",
    
    # 腺体与脂肪细胞 (红色系)
    "ClH23_gland": "#FF9B8C",
    "ClH23_lipophil": "#d62728",
    
    # 过渡态细胞 (黑色)
    "ClH23_trans": "#000000",
    
    # 未知类群 (深灰色)
    "ClH23_unknown_1": "#707070"
}

# 2. 确保 CellType 是 categorical 类型并获取其顺序
# 使用 adata_ClH23_og 对象
adata_ClH23_og.obs['CellTypes'] = adata_ClH23_og.obs['CellTypes'].astype('category')
categories = adata_ClH23_og.obs['CellTypes'].cat.categories

# 3. 按照类别顺序生成颜色列表
new_colors_clh23 = [clh23_color_dict.get(cat, "#cccccc") for cat in categories]

# 4. 写入 adata 对象的 uns 属性
adata_ClH23_og.uns['CellTypes_colors'] = new_colors_clh23

# 5. 绘图
sc.pl.umap(
    adata_ClH23_og, 
    color=['CellTypes'], 
    title="Cladotrichoplax (ClH23) Cell Types", 
    legend_loc='on data',
    frameon=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


fig, ax = plt.subplots(dpi=300)

sc.pl.umap(
    adata_ClH23_og,
    color='CellTypes',
    ax=ax,
    show=False,
    legend_loc=None,   # ⭐ 不显示图例
    frameon=False      # ⭐ 去掉坐标轴边框
)

# ⭐ 去掉坐标刻度与标签、标题
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

output_path = fig_dir + "/33.UMAP.ClH23.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
adata_ClH23_og

In [ ]:
adata_ClH23_og.obs['CellTypes'].value_counts()

In [ ]:
# 保存adata数据
raw_ClH23_path = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/ClH23.OG.normalized.h5ad"
adata_ClH23_og.write(raw_ClH23_path)

In [ ]:
# 导出基因id
adata_ClH23_og.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/ClH23.OG.genes.txt', index=False, header=False)

### 2.9 海绵

In [ ]:
# 海绵
print(adata_Spla)
print(adata_Spla_og)

In [ ]:
# 只保留CellType
adata_Spla_og.obs = adata_Spla_og.obs[['CellTypeFamily']].copy()
# 只保留 X_umap（如果有多维的obsm，需手动清理其它项）
adata_Spla_og.obsm = {'X_umap': adata_Spla_og.obsm['X_umap'].copy()}
adata_Spla_og

In [ ]:
adata_Spla_og.obs['CellTypes'] = 'Spla_' + adata_Spla_og.obs['CellTypeFamily'].astype(str)

In [ ]:
# 清空
adata_Spla_og.uns = {}
adata_Spla_og

In [ ]:
# 修改细胞名
adata_Spla_og.obs.index.name = "CellName"
adata_Spla_og.obs

In [ ]:
# 修改基因索引名
adata_Spla_og.var.index.name = "OrthoGene"
adata_Spla_og.var

In [ ]:
sc.pl.umap(adata_Spla_og, color=['CellTypes'])

In [ ]:
import scanpy as sc

# 1. 定义 Spla 的颜色字典
spla_color_dict = {
    # 结构/上皮类细胞 (紫色)
    "Spla_Endymocytes": "#9467bd",
    
    # 类神经细胞 (绿色) - 对应其他物种的 Neural/Peptidergic
    "Spla_Amoeboid-Neuroid": "#2ca02c",
    
    # 分泌/肽能相关细胞 (红色)
    "Spla_Peptidocytes": "#d62728",
    
    # 原始细胞/干细胞及其近缘 (黄色)
    "Spla_Archeocytes and relatives": "#fedb61",
    
    # 过渡态细胞 (灰色)
    "Spla_transitional": "#707070"
}

# 2. 确保 CellType 是 categorical 类型
# 使用 adata_Spla_og 对象
adata_Spla_og.obs['CellTypes'] = adata_Spla_og.obs['CellTypes'].astype('category')
categories = adata_Spla_og.obs['CellTypes'].cat.categories

# 3. 按照类别顺序生成颜色列表
new_colors_spla = [spla_color_dict.get(cat, "#cccccc") for cat in categories]

# 4. 写入 adata 对象的 uns 属性
adata_Spla_og.uns['CellTypes_colors'] = new_colors_spla

# 5. 绘图
sc.pl.umap(
    adata_Spla_og, 
    color=['CellTypes'], 
    title="Spongilla (Spla) Cell Types", 
    legend_loc='on data',
    frameon=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


fig, ax = plt.subplots(dpi=300)

sc.pl.umap(
    adata_Spla_og,
    color='CellTypes',
    ax=ax,
    show=False,
    legend_loc=None,   # ⭐ 不显示图例
    frameon=False      # ⭐ 去掉坐标轴边框
)

# ⭐ 去掉坐标刻度与标签、标题
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

output_path = fig_dir + "/34.UMAP.Spla.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
adata_Spla_og

In [ ]:
adata_Spla_og.obs['CellTypes'].value_counts()

In [ ]:
# 保存adata数据
raw_Spla_path = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Spla.OG.normalized.h5ad"
adata_Spla_og.write(raw_Spla_path)

In [ ]:
# 导出基因id
adata_Spla_og.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Spla.OG.genes.txt', index=False, header=False)